# Practice notebook for hypothesis tests using NHANES data

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import statsmodels.api as sm
import numpy as np

df = pd.read_csv("nhanes_2015_2016.csv")

In [2]:
df.columns

Index(['SEQN', 'ALQ101', 'ALQ110', 'ALQ130', 'SMQ020', 'RIAGENDR', 'RIDAGEYR',
       'RIDRETH1', 'DMDCITZN', 'DMDEDUC2', 'DMDMARTL', 'DMDHHSIZ', 'WTINT2YR',
       'SDMVPSU', 'SDMVSTRA', 'INDFMPIR', 'BPXSY1', 'BPXDI1', 'BPXSY2',
       'BPXDI2', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXLEG', 'BMXARML', 'BMXARMC',
       'BMXWAIST', 'HIQ210'],
      dtype='str')

In [3]:
df.shape

(5735, 28)

## Question 1

Conduct a hypothesis test (at the 0.05 level) for the null hypothesis that the proportion of women who smoke is equal to the proportion of men who smoke.

In [4]:
# insert your code here
df1 = df.query("RIAGENDR==1")
df2 = df.query("RIAGENDR==2")

n1 = len(df1)
n2 = len(df2)

x1 = len(df1.query("SMQ020==1"))
x2 = len(df2.query("SMQ020==1"))

p1 = x1/n1
p2 = x2/n2

sm.stats.test_proportions_2indep(count1=x1,
                                 nobs1=n1,
                                 count2=x2,
                                 nobs2=n2,
                                 alternative="two-sided",
                                 value=0,
                                 compare="diff",
                                 method="wald",
                                 return_results=True
                                )

Proportions2indepTestResult(statistic=np.float64(16.333437105490432), pvalue=np.float64(5.707620019127988e-60), compare='diff', method='wald', diff=np.float64(0.20770659659296842), ratio=np.float64(1.6822680258947837), odds_ratio=np.float64(2.3984973874024575), variance=np.float64(0.00016171317729646556), alternative='two-sided', value=0, prop1_null=None, prop2_null=None)

Insight : p value is extremely small. With 95% confidence, the proportions of smokers between men and women are estimated to be different.

## Question 2

Partition the population into two groups based on whether a person has graduated college or not, using the educational attainment variable [DMDEDUC2](https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#DMDEDUC2).  Then conduct a test of the null hypothesis that the average heights (in centimeters) of the two groups are equal.  Next, convert the heights from centimeters to inches, and conduct a test of the null hypothesis that the average heights (in inches) of the two groups are equal.

In [5]:
# insert your code here
df1 = df.query("DMDEDUC2==5")["BMXHT"].dropna()
df2 = df.query("DMDEDUC2!=5")["BMXHT"].dropna()

t, p, dof = sm.stats.ttest_ind(df1, df2, alternative="two-sided", value=0, usevar="unequal")
print("t: ", t)
print("p value: ", p)
print("degrees of freedom: ", dof)

t:  7.3655933092491495
p value:  2.4270402820868753e-13
degrees of freedom:  2352.8774252314834


Insight : p value is extremely small. With 95% confidence, the average heights of people who are / are not college graduates are estimated to differ.

## Question 3

Conduct a hypothesis test of the null hypothesis that the average BMI for men between 30 and 40 is equal to the average BMI for men between 50 and 60.  Then carry out this test again after log transforming the BMI values.

In [6]:
# insert your code here
df1 = df.query("RIAGENDR==1 and RIDAGEYR >= 30 and RIDAGEYR <= 40")["BMXBMI"].dropna()
df2 = df.query("RIAGENDR==1 and RIDAGEYR >= 50 and RIDAGEYR <= 60")["BMXBMI"].dropna()

t, p, dof = sm.stats.ttest_ind(df1, df2, alternative="two-sided", value=0, usevar="unequal")
print("t: ", t)
print("p value: ", p)
print("degrees of freedom: ", dof)

t:  0.8980221879527182
p value:  0.36939747833584996
degrees of freedom:  966.0343222922211


Insight : p value is large. We cannot support, unless the confidence level is below 65%, the average BMIs of men who are either 30 to 40 year olds or 50 to 60 year olds differ.

## Question 4

Suppose we wish to compare the mean BMI between college graduates and people who have not graduated from college, focusing on women between the ages of 30 and 40.  First, consider the variance of BMI within each of these subpopulations using graphical techniques, and through the estimated subpopulation variances.  Then, calculate pooled and unpooled estimates of the standard error for the difference between the mean BMI in the two populations being compared.  Finally, test the null hypothesis that the two population means are equal, using each of the two different standard errors.

In [7]:
# insert your code here
df1 = df.query("RIAGENDR==2 and DMDEDUC2==5 and RIDAGEYR >= 30 and RIDAGEYR <= 40")["BMXBMI"].dropna()
df2 = df.query("RIAGENDR==2 and DMDEDUC2!=5 and RIDAGEYR >= 30 and RIDAGEYR <= 40")["BMXBMI"].dropna()

print(sm.stats.ttest_ind(df1, df2, alternative="two-sided", value=0, usevar="unequal"))
print(sm.stats.ttest_ind(df1, df2, alternative="two-sided", value=0, usevar="pooled"))

(np.float64(-4.833662355599096), np.float64(1.962057858215444e-06), np.float64(373.16918356952874))
(np.float64(-4.6439084509564195), np.float64(4.329847689653961e-06), np.float64(523.0))


Insight : p value is extremely small. With 95% confidence, the average BMIs of women of 30 to 40 year olds who are / are not college graduates are estimated to differ.

## Question 5

Conduct a test of the null hypothesis that the first and second diastolic blood pressure measurements within a subject have the same mean values.

In [8]:
# insert your code here
subset = df[['BPXDI1', 'BPXDI2']].dropna()
diff = subset['BPXDI1'] - subset['BPXDI2']

z, p = sm.stats.ztest(diff, value=0)
print("z: ", z)
print("p value: ", p)

z:  4.140387087476751
p value:  3.467202439743513e-05


Insight : p value is extremely small. With 95% confidence, the average diastolic blood pressure measurements of two tests are estimated to differ.